<a href="https://colab.research.google.com/github/ga4gh/analytics-dashboard/blob/develop/notebooks/Analytics_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Analytics Dashboard Overview
This analytics dashboard proof of concept provides an integrated, evidence-based view of GA4GH’s software ecosystem and scientific impact. By combining data from PyPI, GitHub, and PubMed, it supports strategic governance, technical program management, community adoption, and global policy advocacy. The dashboard enables GA4GH to monitor ecosystem health, assess return on investment in standards development, and demonstrate real-world uptake across research and clinical domains.

This notebook is the MVP for a comprehensive analytics dashboard that we plan to release in 2026.

We will use community feedback to iterate on the design and metrics captured here, and will surface enhancements into a final dashboard which will be hosted on the GA4GH website.

If you have ideas for improvements on the key visualisations or metrics shown here, please create an issue directly into the GA4GH Analytics Dashboard repo that describes what you'd like to see on the final dashboard. The website dashboard will be showcased at Connect 2026, so please **contribute your views by January 15th 2026**.

# How this notebook works

This notebook is designed to do three things:
1. **Fetches** data from a new GA4GH Tech Team database that contains the following:
     - **GA4GH-managed GitHub repositories**.  Currently this is representing repos belonging to the GA4GH Github Organisation only (including both GA4GH standards and implementations). This does NOT include external implementations or GA4GH standards that are managed outside the organisation.
     - **PyPi libraries**. Based on a query of the PyPi API that searches for "GA4GH" and was subsequently manually curated to include the following:
        - Packages that come from GA4GH directly (standards)
        - Packages that utilise GA4GH standards (GA4GH as well as external implementations)
     - **Pubmed articles**. Based on a query of the Pubmed API that searches for either "GA4GH" or "Global Alliance for Genomics and Health" which omits entries that refer to a publication correction (erratum), or preprints, as these lead to frequent duplicate entries for the same publication.
1. **Analyzes the data** to compute simple statistics and insights on the three above listed data types.
1. **Creates interactive visualizations** to graphically represent trends in the data.

## How to Run This Notebook on Google Colab (no installation required)

1. **Click the "Open in Colab" button** at the very top of this notebook
1. **Click "Run All"** to execute the full notebook end to end
1. **Wait for results** as it might take a couple of minutes for the notebook to fetch the data and create the analytics
1. **Scroll through the results** - Each section provides different insights



## Useful information on data sourcing

Directly querying the APIs of the three data sources can lead to a number of unpredictable issues, including reaching a quota limit, servers not responding, and changes to the API and the data they provide. To circumvent these issues and guarantee a stable and uniform access to data, the tech team has curated data from the three sources (currently: GitHub, Pypi and Pubmed) and stored them in a database hosted on AWS. Data are then fetched in this notebook via a unified set of APIs, described in more detail [here]('Analytics_Dashboard_Technical_Documentation.ipynb').

<font color='red' size=4>
Data in this current notebook version were last fetched between <b>1st October</b> (GitHub and PyPi) and <b>1st December</b> (PubMed) <b>2025</b>.
</font>

## What you can customize

- Can modify visualization types, styling, and data presented
- Requires knowledge of Plotly and Dash libraries

## How you can contribute

Suggest any changes or feedback by creating an issue in the analytics dashboard repository [here](https://github.com/ga4gh/analytics-dashboard) or send us an email at ga4gh-tech-team@ga4gh.org by January 15th, 2026. You can suggest:
- other ways to analyse, present, or interpret the data
- data and insights you might want to see in the future
- different or additional query terms for PubMed

## What's next on the roadmap
- switch from Pubmed to Europe PMC, which also provides data on preprints, grants, etc
- Incorporate community feedback
- Release version 1 of the web-based analytics dashboard by Connect 2026.
- Explore adding other data types
   
## Important notes

- **Processing time** - Initial run may take 2-3 minutes
- **Internet required** - Fetches data from API set up by the tech team
- **No data storage** - Results are temporary unless you save them

## Troubleshooting

**If visualizations don't appear:**
1. Ensure you have the required libraries installed
1. Try refreshing your browser
1. Check that JavaScript is enabled

**If results seem incomplete:**  
1. Try running individual sections instead of all at once

***Note: This is not a complete dataset. This is a POC and will be more complete in the future***



# Setup


## Libraries required to run this notebook

- `dash`: Used to build interactive dashboard components such as graphs, layouts, and user inputs directly inside the notebook.
- `requests`: Connects to the Analytics Dashboard DB which is curated by the Tech Team and retrieves data in JSON format.
- `pandas`: Helps convert API responses into DataFrames for easy filtering, analysis, and visualization.
- `numpy`: For numerical operations and array/matrix manipulation
- `json`: Used for handling and inspecting raw JSON responses.
- `typing` (List, Optional, Dict, Any): Adds optional type hints to improve code readability and maintainability.
- `plotly.express`: Creates interactive charts (bar, line, pie, scatter) quickly and easily.

These libraries are widely used and are usually already present by default on Google Colab, with the exception of `dash`, which we will need to make sure is installed before proceeding to running the rest of this notebook.

First, we install the `dash` package.

In [1]:
!pip install dash


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip


Now that `dash` is installed, we can load all the rest of the packages we need for this notebook.

In [2]:
import requests
import json
import dash
from typing import List, Optional, Dict, Any
from dash import Dash, dcc, html, Input, Output, callback, dash_table, jupyter_dash
from dash.dependencies import Input, Output
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
from collections import Counter
from datetime import datetime
import numpy as np



## Configurations

There are several API endpoints available for the dashboard and details are provided [here]('Analytics_Dashboard_Technical_Documentation.ipynb') in a separate notebook containing the technical documentation for this project. Each of the three data sources has one or more endpoints, serving specific slices or views of the data. All of them return a JSON payload.

Here we will declare some utility functions and variables that later on will be used to access the data.

In [3]:
# Base URL
BASE_URL = "http://analytics-staging.ga4gh.org:8000"

# Github Endpoints
REPO_ENDPOINT = f"{BASE_URL}/github/name/" # fetches repo based on repo name parameter provided by user after name/
REPO_OWNER_ENPOINT = f"{BASE_URL}/github/owner/" # fetches repos based on repo owner parameter provided user after owner/
ALL_REPOS_ENDPOINT = f"{BASE_URL}/github/all"

# PyPi Endpoints
TOTAL_PACKAGES_ENDPOINT = f"{BASE_URL}/pypi/total-packages"
PACKAGE_VERSIONS_ENDPOINT = f"{BASE_URL}/pypi/package-versions"
RELEASES_OVER_YEARS_ENDPOINT = f"{BASE_URL}/pypi/releases-over-years"
PYPI_DETAILS_ENDPOINT = f"{BASE_URL}/pypi/project-details"
FIRST_RELEASES_ENDPOINT = f"{BASE_URL}/pypi/first-releases"
ALL_PACKAGES_ENDPOINT = f"{BASE_URL}/pypi/all-packages"

# PubMed Endpoints
PM_KEYWORD = f"{BASE_URL}/pubmed/articles/GA4GH"

# Function that will take a list of repos and retrieve them from the DB
def get_repos(endpoint: str, list_of_repos: List[str]) -> List[Dict[str, Any]]:

    items: List[Dict[str, Any]] = []
    url = endpoint

    for repo in list_of_repos:
        resp = requests.get(url + repo)
        resp.raise_for_status()
        items.append(resp.json()[0])

    return items

# Function to make a request to the endpoints to return data from the DB addressing any pagination issues
def get_json(endpoint: str, token: Optional[str] = None, per_page: int = 100) -> List[Dict[str, Any]]:

    headers = {}
    if token:
        headers["Authorization"] = f"token {token}"

    params = {"per_page": per_page, "page": 1}
    items: List[Dict[str, Any]] = []
    url = endpoint

    while True:
        resp = requests.get(url, headers=headers, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()

        if not isinstance(data, list):
            return data

        items.extend(data)

        if "next" in resp.links:
            url = resp.links["next"]["url"]
            params = None
        else:
            break

    return items


def get_last_year(dates):
    last_year = []
    for d in dates:
        if hasattr(d, "year"):
            y = d.year
        else:
            y = datetime.fromisoformat(d).year
        if y == 2025:
            last_year.append(d)
    return last_year

We can now access the data and load the returned JSON file into a pandas DataFrame, for ease of use later on.

In [4]:
#PubMed
pubmed_json = get_json(PM_KEYWORD)
pm_df = pd.DataFrame.from_records(pubmed_json)
pm_df['publish_date'] = pd.to_datetime(pm_df['publish_date'], utc=True, errors='raise')
pm_df['year'] = pm_df['publish_date'].dt.year
pm_last_year = get_last_year(pm_df['publish_date'])

# GitHub
GA4GH_json = get_json(ALL_REPOS_ENDPOINT)
gh_df = pd.DataFrame.from_records(GA4GH_json)
gh_df['last_updated'] = pd.to_datetime(gh_df['last_updated'], utc=True, errors='raise')
gh_df['pushed_at'] = pd.to_datetime(gh_df['pushed_at'], utc=True, errors='raise')
gh_df['created_on'] = pd.to_datetime(gh_df['created_on'], utc=True, errors='raise')
gh_last_year = get_last_year(gh_df['created_on'])

# PyPi
pypi_packages_json = get_json(PYPI_DETAILS_ENDPOINT)
pypi_df = pd.DataFrame.from_records(pypi_packages_json)

pypi_first_versions_df = pd.DataFrame.from_records(get_json(FIRST_RELEASES_ENDPOINT))

pypi_df = pypi_df.merge(pypi_first_versions_df,how='inner',on='project_name')
assert pypi_df.shape[0] == int(get_json(TOTAL_PACKAGES_ENDPOINT).get("total_packages", 0))
pypi_df['first_release'] = pd.to_datetime(pypi_df['release_date'], utc=True, errors='raise')

pypi_last_year = get_last_year(pypi_df['first_release'])



##  Figure 1: Total impact


This figure provides a **high-level summary of GA4GH’s overall ecosystem impact** by bringing together three core dimensions of activity:

- **Software development and implementation** (from GitHub)
- **Scientific research and publications** (from PubMed)
- **Standards-enabled software distribution** (from PyPI)

Rather than focusing on one platform in isolation, this figure acts as an **executive snapshot of the full GA4GH value chain** — from standards implementation, to community adoption, to scientific and clinical impact.

Depending on how the summary is displayed in this chart (e.g., aggregated counts, indicators, or combined metrics), it is intended to answer one central question:

> *How large and active is the GA4GH ecosystem right now?*

---

**How to read this plot**

- Each component of the figure represents a **different evidence stream**:
  - GitHub reflects **engineering and standards implementation activity**
  - PubMed reflects **research translation and scientific uptake**
  - PyPI reflects **practical software distribution and reuse**
- Larger values indicate:
  - More active development,
  - Broader scientific visibility,
  - And greater real-world software adoption.

This figure should be read as a **context-setting overview**, not a detailed technical diagnostic.

---


In [5]:
def plot_indicators():

    fig = go.Figure()

    fig = make_subplots(
        rows=1, cols=3,
        specs=[[{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}]],
        horizontal_spacing=0.08

    )
    fig.add_trace(
        go.Indicator(mode="number+delta", value=pm_df.shape[0], title={"text": "PubMed Articles"}, delta={"reference": pm_df.shape[0] - len(pm_last_year)}),
        row=1, col=3
    )
    fig.add_trace(
        go.Indicator(mode="number+delta", value=gh_df.shape[0], title={"text": "GitHub Repositories"}, delta={"reference": gh_df.shape[0] - len(gh_last_year)}),
        row=1, col=1
    )
    fig.add_trace(
        go.Indicator(mode="number+delta", value=pypi_df.shape[0], title={"text": "PyPi Packages"}, delta={"reference": pypi_df.shape[0] - len(pypi_last_year)}),
        row=1, col=2

    )

    fig.update_layout(
        template="simple_white",
        title={
            "text": "Figure 1: Total GA4GH Data Points Per Source With 2025 Increases\n",
            "x": 0.5,
            "xanchor": "center",
            "font": {"size": 26},
            "pad": {"b": 40},
        },
        margin=dict(l=20, r=20, t=50, b=20),
        height=300,
        width=1000
    )
    fig.show()
plot_indicators()


**How to interpret**

Figure 1 demonstrates how GA4GH standards move through the full lifecycle:

1. **Standards become software** (GitHub)
2. **Software becomes reusable infrastructure** (PyPI)
3. **Infrastructure enables research and clinical studies** (PubMed)

Growth across all three indicates a **healthy, functioning international standards ecosystem**. If one dimension grows while another stagnates, that can signal:
- a bottleneck in translation,
- a documentation or adoption barrier,
- or a need for targeted community support.

---

> **Key Takeaway (Policy & Strategy):**  
> This figure provides a single, integrated view of GA4GH’s global footprint across software, infrastructure, and science, allowing decision-makers to assess the overall return on investment in GA4GH standards at a glance.


## Figure 2: Growth of GA4GH resources over time

**What this plot shows**

This figure shows how key GA4GH-related resources have **grown over time**, based on two independent evidence streams:

- **GitHub repositories** — representing software development and standards implementation activity  
- **PubMed publications** — representing scientific research and translational uptake  

Each year on the timeline shows how many:
- Repositories were newly created or updated on GitHub, and  
- GA4GH-related articles were published in the scientific literature.

Together, these trends show how GA4GH has evolved **both technically and scientifically over time**.

---

**How to read this plot**

- The **x-axis** represents calendar years.
- The **y-axis** represents the number of resources (repositories or publications) associated with GA4GH in each year.
- Separate lines (or grouped bars, depending on display) distinguish:
  - GitHub activity  
  - PubMed publication activity  

Rising lines indicate periods of growth, while flatter or declining sections indicate slower expansion or consolidation.

---




In [6]:
to_combine_pm_df = pd.DataFrame({'item': pm_df['title'],
                                 'year': pm_df['year'],
                                 'Source': 'PubMed Articles'})

to_combine_gh_df = pd.DataFrame({'item': gh_df['name'],
                                 'year': gh_df['created_on'].dt.year,
                                 'Source': 'GitHub Repos'})

to_combine_pypi_df = pd.DataFrame({'item': pypi_df['project_name'],
                                   'year': pypi_df['first_release'].dt.year,
                                   'Source': 'PyPi Packages'})

combined_year_df = pd.concat([to_combine_pm_df,to_combine_gh_df,to_combine_pypi_df])

In [7]:
# Group by Year and Source
year_plot_df = combined_year_df.groupby(['year', 'Source']).agg({'item': list}).reset_index()
year_plot_df = year_plot_df.sort_values(['Source', 'year'])

# List items and calculate cumulative sum
year_plot_df['items_str'] = year_plot_df['item'].apply(lambda items: '<br>'.join(items))
year_plot_df['yearly_count'] = year_plot_df['item'].apply(len)
year_plot_df['yearly_cumulative_count'] = year_plot_df.groupby(['Source'])['yearly_count'].cumsum()

In [8]:
def plot_year():
    fig = px.line(
        year_plot_df,
        x='year',
        y='yearly_cumulative_count',
        color='Source',
        markers=True,
        title='Figure 2: Growth of GA4GH Resources Over Time',
        labels={'year': 'Year', 'yearly_cumulative_count': 'Total to date'},
        custom_data=['items_str', 'Source', 'yearly_count'],
        template='simple_white'
    )

    # 5. Update Hover Template
    fig.update_traces(
        hovertemplate=(
            '<b>%{customdata[1]}</b><br>' # Source Name
            'Year: %{x}<br>'
            'Number of new items: %{customdata[2]}<br><br>'
            '<b>New Items:</b><br>%{customdata[0]}<extra></extra>'
        ),
        marker=dict(size=8),
    )

    fig.update_layout(
        xaxis_title='Year',
        yaxis_title='Cumulative number of new items',
        title_font_size=26,
        margin=dict(l=40, r=20, t=70, b=120),
        height=600,
        legend_title_text='Resource',
        width=1080
    )
    fig.show()
plot_year()

**How to interpret:**

This figure illustrates the **long-term momentum of the GA4GH ecosystem**:

- Growth in **GitHub activity** reflects:
  - expanding engineering capacity,
  - new standards implementations,
  - and increased tooling to support interoperability.
- Growth in **PubMed publications** reflects:
  - wider scientific adoption,
  - maturing use cases,
  - and translation into real-world research and clinical studies.

When these curves grow together, it indicates successful **flow from standards → software → science**.  
If they diverge, it can signal:
- adoption lag,
- documentation or usability barriers,
- or the need for targeted outreach and training.

---

> **Key Takeaway (Policy & Strategy):**  
> Sustained growth across both software development and scientific publication demonstrates that GA4GH standards are not only being built, but are also being actively adopted and used across the global research ecosystem.

## Figure 3: PubMed (Top 10 Journals)


**What this plot shows**

This figure shows the **top 10 scientific journals publishing the highest number of GA4GH-related articles**, based on PubMed indexing. Rather than showing all GA4GH-related publications across the entire literature, this view highlights the **journals where GA4GH work is most concentrated**.

Each bar (hover to see list of individual publications) represents:
- One journal, and  
- The number of GA4GH-related articles published in that journal.

This provides a **targeted view of where GA4GH is most visible in the scientific publishing ecosystem**.

---

**How to read this plot**

- The **x-axis** represents journal titles.
- The **y-axis** shows the number of GA4GH-related articles published in each journal.
- Taller bars indicate journals that publish GA4GH-related work more frequently.
- Shorter bars indicate journals with occasional GA4GH-related publications.

This visualization focuses on **distribution across journals**, not changes over time.

---




In [9]:
articles_in_top = []

for j, g in pm_df.groupby('journal'):
    titles_sorted = g.sort_values('title')
    details = "<br>".join(r['title'] for _, r in titles_sorted.iterrows())
    articles_in_top.append({'journal': j, 'Count': len(g), 'Details': details})

top_journals = pd.DataFrame(articles_in_top).sort_values('Count', ascending=False).head(10).reset_index(drop=True)

In [10]:
def plot_top_journals():

    if isinstance(top_journals, pd.Series):
        df = top_journals.reset_index()
        df.columns = ["journal", "Count"]
    else:
        df = top_journals.copy()

    df = df.sort_values("Count", ascending=False).reset_index(drop=True)

    # build customdata for hover (Count + Details if available)
    if "Details" in df.columns:
        customdata = df[["Count", "Details"]].values
        hovertemplate = "Count: %{customdata[0]}<br>%{customdata[1]}<extra></extra>"
    else:
        customdata = df[["Count"]].values
        hovertemplate = "Count: %{customdata[0]}<extra></extra>"

    fig = go.Figure(
        go.Bar(
            x=df["journal"],
            y=df["Count"],
            #marker_color=colors,
            customdata=customdata,
            hovertemplate=hovertemplate,
            marker_line_width=0,

        )
    )

    fig.update_layout(
        title="Figure 3: Top 10 Journals for GA4GH Publications",
        title_font_size=26,
        barmode="relative",
        xaxis_tickangle=-45,
        margin=dict(l=40, r=20, t=80, b=150),
        xaxis_title="Journal",
        yaxis_title="Count",
        legend_title_text=" ",
        height=600,
        template="simple_white",
        width=1000
    )

    fig.show()
plot_top_journals()

**How to interpret**

This figure helps answer the question:

> *Where is GA4GH work most visible in the scientific literature?*

High counts in specific journals indicate:
- Strong alignment between GA4GH standards and the focus of those research communities,
- Established citation and publication pathways for GA4GH-enabled work,
- Key venues for outreach, special issues, and community engagement.

Lower representation in other journals does **not** imply lack of impact overall — it simply reflects that this figure is intentionally scoped to the **top 10 most active journals**.

---

> **Key Takeaway (Policy & Strategy):**  
> GA4GH-related research is concentrated in a small number of high-visibility journals, highlighting priority venues for strategic partnerships, outreach, and dissemination of standards-based research.


## Figure 4: Activity status of GA4GH GitHub repositories


**What this plot shows**

This figure classifies GA4GH GitHub repositories based on their **recent development activity**. Repositories are grouped into activity categories (for example: recently active, moderately active, or inactive), based on the timing of their most recent updates.

Rather than focusing on popularity or size, this figure answers a different question:

> *Which GA4GH repositories are actively being worked on right now, and which are dormant?*

This makes it a direct indicator of **software maintenance health** across the GA4GH ecosystem.

---

**How to read this plot**

- Each bar or segment represents a **group of repositories** that fall into a specific activity category.
- The **height or size** reflects how many repositories belong to each category.
- Categories typically progress from:
  - active (with recent updates in the last year),
  - to moderate activity (with updates in the last 3 years),
  - to inactive (no updates for a long period - over 3 years)
  - to archived (no longer being updated).

This view is about **recency of work happening on a given GA4GH product**, not how widely a repository is used.

---




In [11]:
fetch_date = '2025-10-01'
target_date = pd.to_datetime(fetch_date, utc=True)

gh_df['days_since_pushed_at'] = (target_date - gh_df['pushed_at']).dt.days
gh_df['days_since_last_updated'] = (target_date - gh_df['last_updated']).dt.days

gh_df['activity_score'] = 1/(1+gh_df['days_since_pushed_at']) + 1/(1+gh_df['days_since_last_updated'])

gh_activity_df = gh_df.sort_values('activity_score', ascending=False).head(15).reset_index(drop=True)

In [12]:
conditions = [
    (gh_df['is_archived'] == False) & (gh_df['days_since_pushed_at'] < 365),
    (gh_df['is_archived'] == False) & (gh_df['days_since_pushed_at'] >= 365) & (gh_df['days_since_pushed_at'] <= 1095),
    (gh_df['is_archived'] == False) & (gh_df['days_since_pushed_at'] > 1095),
    (gh_df['is_archived'] == True)
]
choices = ['Active', 'Moderate activity', 'Inactive', 'Archived']

gh_df['activity_status'] = np.select(conditions, choices, default='Unknown')

In [13]:
gh_activity_counts = gh_df['activity_status'].value_counts().reset_index()
gh_activity_counts.columns = ["Category", "Count"]

def plot_github_status():

    fig = go.Figure(data=[
        go.Pie(
            labels=gh_activity_counts["Category"],
            values=gh_activity_counts["Count"],
            hole=0.4,
            textinfo="label+percent",
            hoverinfo="label+value+percent"
        )
    ])

    fig.update_layout(
        title={
            "text": "Figure 4: Activity Status of the GA4GH GitHub Repositories",
            "x": 0.5,         
            "xanchor": "center",
            "yanchor": "top",
            "font": {"size": 26, "color": "#2C3E50"}
        },
        plot_bgcolor="#f9f9f9",
        legend_title_text='Activity Status',
        paper_bgcolor="#ffffff",
        width=1000,
        height=600
    )

    fig.show()
plot_github_status()

**How to interpret:**

This figure provides insight into the **sustainability of GA4GH software assets**:

- A large proportion of **active repositories** suggests:
  - healthy ongoing maintenance,
  - responsive development teams,
  - and robust technical stewardship.
- A large proportion of **inactive repositories** may indicate:
  - completed (stable) or sunset projects,
  - loss of maintainer capacity,
  - or potential technical debt.

For GA4GH leadership, this figure helps identify:
- where sustained funding or staffing may be required,
- which projects may need revitalisation or formal retirement,
- and areas where community contribution could be encouraged.

---

> **Key Takeaway (Policy & Strategy):**  
> This figure provides a direct measure of the operational health of GA4GH’s software ecosystem, supporting informed decisions about which projects require sustained investment, consolidation, or transition planning.

## Figure 5: Most recently updated GA4GH GitHub repositories

**What this plot shows**

This figure highlights the **GA4GH GitHub repositories that were updated most recently**, as of **October 1, 2025**. It provides a snapshot of **where active development is happening right now** within the GA4GH software ecosystem.

Each bar (or row, depending on the format) represents a single repository, ranked by:
- The date of its most recent code update (`pushed_at` on GitHub).

This view focuses strictly on **recency of development**, not on popularity or long-term impact.

---

**How to read this plot**

- Repositories are ordered from the **most recently updated** to less recently updated.
- The position of a repository in the ranking shows how recently developers have contributed code.
- This allows you to quickly identify which projects currently have **active engineering attention**.

This is especially useful as a **real-time operational monitoring view**.

---



In [14]:
def plot_gh_activity():
    fig = px.bar(
        gh_activity_df,
        x='name',
        y='activity_score',
        labels={'activity_score': 'Count', 'name': 'Repo'},
        title="Figure 5: Most Recently Updated GA4GH-Related Repositories",
        custom_data=['days_since_pushed_at', 'days_since_last_updated'],
    )

    fig.update_traces(
        hovertemplate='Days since last commit: %{customdata[0]}<br>Days since last repo update: %{customdata[1]}<extra></extra>',
        marker_line_width=0
    )

    fig.update_layout(
        barmode='relative',
        xaxis_tickangle=-45,
        margin=dict(l=40, r=20, t=80, b=150),
        xaxis_title='Repository',
        yaxis_title='Recent activity (arbitrary unit)',
        legend_title_text='',
        title_font_size=26,
        height=700,
        width=1000,
        template='simple_white',
    )

    fig.show()

plot_gh_activity()

**How to interpret**

Repositories appearing at the top of this list typically represent:

- Active standards implementations,
- Ongoing development of tooling and infrastructure,
- Projects responding to new requirements, bug reports, or community feedback.

Repositories that fall far down the list may be:
- Stable and in maintenance-only mode,
- Temporarily dormant due to staffing or funding gaps,
- Or near end-of-life.

This figure complements:
- **Activity status of the repos** (Figure 4, longer-term maintenance health),
- **Interest Metrics for GA4GH Repositories** (Figure 6, external demand and visibility).

Together, they provide a balanced view of:
> *Where development effort is currently being applied versus where community demand exists.*

---

> **Key Takeaway (Policy & Strategy):**  
> This figure identifies the GA4GH software assets that currently have active developer engagement, helping leadership and funders to understand where short-term engineering capacity is being deployed.


## Figure 6: Interest metrics for GA4GH Github repositories


**What this plot shows**

This figure summarises **community interest and engagement** with GA4GH GitHub repositories using commonly used GitHub metrics, such as:

- ⭐ Stars — how many users have bookmarked or endorsed a repository  
- 🍴 Forks — how many users have created their own copy to modify or build upon  
- 👀 Watchers — how many users are following updates  
- 🐞 Issues — how many questions, bugs, or feature requests are being actively tracked  

Rather than measuring development activity directly, this figure reflects the **external visibility and perceived value** of GA4GH repositories within the wider open-source and research software communities.

---

**How to read this plot**

- Each bar (or grouped set of bars) represents one GA4GH repository.
- The height of each bar corresponds to one interest metric (e.g., stars or forks).
- Taller bars indicate higher levels of community engagement and attention.

This plot allows you to quickly compare which repositories are:
- most visible,
- most reused,
- or most actively discussed.

---



In [15]:
gh_df['total_interest'] = gh_df['subscribers_count'] + gh_df['stargazers_count'] + gh_df['forks_count']
gh_interest_df = gh_df.sort_values('total_interest', ascending=False).head(10).reset_index(drop=True)

In [16]:
def plot_gh_interest():
    fig = px.bar(
        gh_interest_df,
        x='name',
        y=['subscribers_count','stargazers_count','forks_count'],
        title='Figure 6: Interest Metrics for GA4GH GitHub Repositories',
        color_discrete_sequence=['#540d6e', '#ee4266', '#ffd23f'],
        template='simple_white'
    )

    fig.update_layout(
        barmode='stack',
        xaxis_tickangle=-45,
        margin=dict(l=40, r=20, t=80, b=150),
        xaxis_title='Repository name',
        yaxis_title='Interest (Subscribers + Stargazers + Forks)',
        legend_title_text='Interaction Type',
        title_font_size=26,
        height = 700,
        width=1080
    )

    fig.update_traces(marker_line_width=0)

    fig.show()
plot_gh_interest()

**How to interpret**

High interest metrics typically indicate that a repository is:

- Widely used as part of downstream tools or pipelines,
- Referenced by external projects,
- Or actively evaluated by the community.

However, high interest does **not always mean active maintenance**, and low interest does **not always mean low importance**. Some infrastructure components may be:
- Mission-critical but relatively invisible,
- Highly stable and therefore rarely discussed.

This figure should therefore be interpreted alongside:
- **Activity status of the repos (Figure 4)**
- **Most recently updated repos (Figure 5)**
to fully understand both **demand and capacity**.

---

> **Key Takeaway (Policy & Strategy):**  
> Community interest metrics help identify the GA4GH repositories with the greatest external visibility and adoption, informing outreach priorities, partnership opportunities, and long-term sustainability planning.



## Figure 7: Functional categorization of GA4GH-related PyPi Packages

**What this plot shows**

This figure shows how GA4GH-related Python packages on PyPI are distributed across different **functional categories**. Each category represents a broad type of capability within the GA4GH software ecosystem, such as:

- Core GA4GH standards
- Implementations of GA4GH standards
- Packages where GA4GH is referenced

Each bar or slice (depending on the chart type) represents:
- One category, and  
- The number of PyPI packages that fall into that category.

This figure provides a **capability-level view** of GA4GH’s software portfolio.

---

**How to read this plot**

- The **x-axis** (or labels) list the different package categories.
- The **y-axis** (or proportional slice size) shows how many packages belong to each category.
- Larger bars or slices indicate areas where GA4GH software development is most concentrated.

This is a **structural view of the ecosystem**, not a time-based view.

---


In [17]:
pypi_type_counts = pypi_df['category'].value_counts().reset_index()
pypi_type_counts.columns = ["Category", "Count"]

def plot_pypi_status():

    fig = go.Figure(data=[
        go.Pie(
            labels=pypi_type_counts["Category"],
            values=pypi_type_counts["Count"],
            hole=0.4,
            textinfo="label+percent",
            hoverinfo="label+value+percent"
        )
    ])

    fig.update_layout(
        title={
         "text": "Figure 7: Category of GA4GH-Related PyPi Packages",
            "x": 0.5,         
            "xanchor": "center",
            "yanchor": "top",
            "font": {"size": 26, "color": "#2C3E50"}
        },
        plot_bgcolor="#f9f9f9",
        paper_bgcolor="#ffffff",
        legend_title_text='Category',
        height=600,
        width=1000
    )

    fig.show()
plot_pypi_status()

**How to interpret**

This figure answers the question:

> *What kinds of software does the GA4GH ecosystem primarily produce and maintain?*

- Categories with a high number of packages indicate:
  - strong community focus,
  - mature standards areas,
  - and well-developed implementation ecosystems.
- Smaller categories may represent:
  - emerging technical domains,
  - under-resourced capability areas,
  - or specialised niche tooling.

For GA4GH leadership and program managers, this figure helps:
- Assess whether software investments align with strategic priorities,
- Identify gaps where new tooling could accelerate adoption,
- Balance future development across standards areas.

---

> **Key Takeaway (Policy & Strategy):**  
> The distribution of packages by category reveals where GA4GH’s software investments are currently concentrated and highlights strategic opportunities to strengthen underrepresented capability areas.


## Figure 8: Most frequently released GA4GH-related PyPi packages

**What this plot shows**

This figure highlights the GA4GH-related Python packages on PyPI that have the **highest release frequency**, restricted to packages that have **at least 90 days of release history**. This filtering step ensures that the figure focuses on packages with **sustained development activity**, rather than one-off or very new projects.

Each bar represents:
- A single PyPI package, and  
- The number of versions released over the qualifying time window.

This figure serves as a proxy for **maintenance intensity and development velocity** across the GA4GH software ecosystem.

---

**How to read this plot**

- The **x-axis** lists the package names.
- The **y-axis** shows the number of releases.
- Taller bars indicate packages that are updated more frequently.

Because only packages with at least 90 days of history are included, the comparison is **more fair and meaningful** than a simple count across all packages.

---



In [18]:
pypi_all_versions_json = get_json(ALL_PACKAGES_ENDPOINT)

In [19]:
rows_list = []

for ver in pypi_all_versions_json:
    dict1 = {'package': ver['package']['project_name'],
             'package_version': ver['version']['package_version'],
             'release_date': pd.to_datetime(ver['version']['release_date'], utc=True, errors='raise'),
             'category': ver['package']['category']
            }

    rows_list.append(dict1)

pypi_all_versions_df = pd.DataFrame(rows_list)

In [20]:
pypi_all_versions_df = pypi_all_versions_df.sort_values(['package','release_date','category']).reset_index(drop=True)
pypi_release_range_df = pypi_all_versions_df.groupby('package')['release_date'].agg(['first','last','count'])
pypi_release_range_df['range_days'] = (pypi_release_range_df['last']-pypi_release_range_df['first']).dt.days
pypi_release_range_df['frequency_year'] = (pypi_release_range_df['count']/pypi_release_range_df['range_days'])*365
category_per_package = pypi_all_versions_df.groupby('package')['category'].first()
pypi_release_range_df = pypi_release_range_df.join(category_per_package)
pypi_release_range_df['first'] = pypi_release_range_df['first'].dt.strftime('%Y-%m-%d')
pypi_release_range_df['last'] = pypi_release_range_df['last'].dt.strftime('%Y-%m-%d')

pypi_release_range_df = pypi_release_range_df.reset_index()

In [21]:
def plot_pypi_frequency():

    df = (
        pypi_release_range_df[pypi_release_range_df["range_days"] > 90]
        .sort_values("frequency_year", ascending=False)
        .head(10)
        .copy()
    )

    color_map = {
        "Implementation": "#1f77b4",
        "GA4GH Standard": "#2ca02c",
        "Reference": "#ff7f0e",
    }

    colors = df["category"].map(color_map).fillna("#7f7f7f").tolist()

    fig = px.bar(
        df,
        x="package",
        y="frequency_year",
        labels={"frequency_year": "Average number of new versions per year", "package": "Package"},
        title="Figure 8: Most Frequently Released GA4GH-Related PyPi Packages",
        custom_data=["count", "first", "last"],
    )

    fig.update_traces(
        marker_color=colors,
        hovertemplate="%{customdata[0]} versions between %{customdata[1]} and %{customdata[2]}<extra></extra>",
        marker_line_width=0,
    )

    used_cats = [c for c in ["Implementation", "GA4GH Standard", "Reference"] if c in df["category"].unique()]
    for cat in used_cats:
        fig.add_trace(
            go.Scatter(
                x=[None],
                y=[None],
                mode="markers",
                marker=dict(size=10, color=color_map.get(cat, "#7f7f7f")),
                name=cat,
                showlegend=True,
            )
        )

    fig.update_layout(
        barmode="relative",
        xaxis_tickangle=-45,
        margin=dict(l=40, r=200, t=80, b=150),  
        xaxis_title="PyPi package name",
        yaxis_title="Average number of new versions per year",
        legend_title_text="Category",
        legend=dict(orientation="v", x=1.02, y=1),
        height=600,
        width=1080,
        template="simple_white",
         title_font_size=26,
        )

    fig.show()

plot_pypi_frequency()

**How to interpret**

Packages with high release frequency typically indicate:

- Active development teams,
- Rapid iteration in response to:
  - bug reports,
  - evolving standards,
  - or community feedback.
- High operational importance in supporting GA4GH standards.

However, very high release frequency can also reflect:
- Fast-moving requirements,
- Intensive maintenance pressure,
- Or the challenges of stabilising complex infrastructure.

For GA4GH leadership, this figure helps identify:
- **Core digital infrastructure** that requires sustained resourcing,
- Projects with unusually high maintenance burden,
- Candidates for formal long-term sustainability planning.

---

> **Key Takeaway (Policy & Strategy):**  
> A small subset of GA4GH packages accounts for a disproportionate share of ongoing software maintenance, underscoring the importance of targeted long-term funding for these high-velocity infrastructure components.

## Figure 9: Searchable Table of GA4GH-related PyPi Packages

**What this table shows**

This searchable data table shows the **total number of GA4GH-related Python packages currently indexed on PyPI**, which is **82 packages** at the time of this analysis.

This represents the **overall scale of the GA4GH Python software ecosystem**, including:
- Core GA4GH standards
- Implementations of GA4GH standards
- Packages where GA4GH is referenced

Unlike the other figures, this is a **size-of-ecosystem indicator**, rather than an activity or growth metric.

---

**How to read this metric**

- The value **82** represents the total count of distinct PyPI packages associated with GA4GH.
- Each row in the table (package) corresponds to at least one distributable software component that can be directly installed and reused by the global community.

This number should be interpreted as a **snapshot in time**, and is expected to increase as new tools and standards implementations are released.

---




In [22]:
from IPython.display import display, HTML
total_packages = len(pypi_df)
# Convert DataFrame to HTML with DataTables
col_to_show = ['project_name','description',
               'author_name','author_email','category']

html_str = pypi_df[col_to_show].to_html(classes="display", table_id="my_table", index=False)
html_template = f"""
<h3>Total PyPI Packages: {total_packages}</h3>
{html_str}
<link rel="stylesheet" type="text/css" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.css">
<script type="text/javascript" charset="utf8" src="https://code.jquery.com/jquery-3.7.1.js"></script>
<script type="text/javascript" charset="utf8" src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.js"></script>
<script>
$(document).ready(function() {{
    $('#my_table').DataTable({{
        "pageLength": 10,
        "lengthMenu": [5, 10, 20, 50],
        "order": [[0, "asc"]],
        "searching": true,
        "info": true,
        "paging": true
    }});
}});
</script>
"""
# Display inline in notebook
display(HTML(html_template))


project_name,description,author_name,author_email,category
data-repo-client,Data Repository API,OpenAPI Generator community,team@openapitools.org,Implementation
cwltool,Common workflow language reference implementation,Common workflow language working group,common-workflow-language@googlegroups.com,Implementation
pyphetools,Generate and work with GA4GH phenopackets,None,"Peter Robinson <peter.robinson@bih-charite.de>, Daniel Danis <daniel.gordon.danis@protonmail.com>",Implementation
planemo,Command-line utilities to assist in building tools for the Galaxy project (http://galaxyproject.org/).,None,Galaxy Project and Community <galaxy-committers@lists.galaxyproject.org>,Implementation
bento-lib,A set of common utilities and helpers for Bento platform services.,David Lougheed,david.lougheed@mail.mcgill.ca,Reference
bycon,A Python-based environment for the GA4GH Beacon genomics API,None,Michael Baudis <m@baud.is>,GA4GH Standard
sapporo,The sapporo-service is a standard implementation conforming to the Global Alliance for Genomics and Health (GA4GH) Workflow Execution Service (WES) API specification.,None,"""DDBJ (Bioinfomatics and DDBJ Center)"" <tazro.ohta@chiba-u.jp>",Implementation
variation-normalizer,VICC normalization routine for variations,"Alex Wagner, Kori Kuzma, James Stevenson",None,Implementation
ga4gh.vrs,GA4GH Variation Representation Specification (VRS) reference implementation,None,"Larry Babb <lbabb@broadinstitute.org>, Reece Hart <reecehart@gmail.com>, Andreas Prlic <andreas.prlic@gmail.com>, Alex Wagner <alex.wagner@nationwidechildrens.org>",GA4GH Standard
nmdc-runtime,A runtime system for NMDC data management and orchestration,None,None,Reference


**How to interpret**

This metric provides a direct measure of **how extensive the GA4GH software footprint has become**:

- A larger number of packages indicates:
  - broader implementation of GA4GH standards,
  - increased diversity of technical use cases,
  - and stronger community engagement in software development.
- Tracking this number over time allows GA4GH to:
  - Measure ecosystem expansion,
  - Demonstrate tangible outputs to funders and policy partners,
  - Monitor the long-term growth of its implementation community.

This metric complements:
- **Category of the packages** (Figure 7; what kinds of tools exist),
- **Most frequently released packages** (Figure 8; what requires sustained maintenance).


---

> **Key Takeaway (Policy & Strategy):**  
> With 82 publicly distributed Python packages, GA4GH has established a substantial and growing global software ecosystem that underpins the real-world implementation of its data-sharing standards.

Together, these figures provide complementary evidence across the full GA4GH value chain: from software development and technical sustainability, through scientific publication and global uptake, to strategic governance and policy impact. Used collectively, they enable GA4GH to make data-driven decisions about standards prioritisation, funding allocation, infrastructure sustainability, and international coordination.
